# Post-Processing Visualization for SWAN Wave Model Output

---

## Overview

After running the SWAN wave model, the output fields need to be visualized for analysis and interpretation. This notebook reads **actual SWAN output data** from the California Bay simulation (produced by the site-selection and optimization workflows) and demonstrates four standard post-processing visualization types:

| Visualization | Data Source | What it shows |
|--------------|-------------|---------------|
| **Temporal** (time series) | TABLE output (`site_output.tab`) | Evolution of $H_s$ and $T_p$ at selected candidate sites |
| **Spatial** (2-D map) | BLOCK output (`ga_block.mat`) | Wave height distribution over the computational grid with direction quivers |
| **Wave Power** (2-D map) | BLOCK output (`ga_block.mat`) | Wave power density $P = \frac{\rho g^2}{64\pi} H_s^2 T_p$ over the domain |
| **Spectral** (polar plot) | TABLE output (`site_output.tab`) | Reconstructed JONSWAP directional spectrum from SWAN parameters |

### Data

The SWAN model was run for the **California Bay** region (Jan–Feb 2024) with:
- **TABLE output** at 391 candidate sites (hourly, 1440 timesteps) — from the AHP-TOPSIS site-selection run
- **BLOCK output** on the full computational grid (217 × 121 cells, 6-hourly, 240 timesteps) — from the GA optimization run

### Dependencies

```
numpy, scipy, matplotlib
```

In [ ]:
import os
import sys
import re
import datetime
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.colors as mcolors

%matplotlib inline

# Import swan_runner utilities for reading SWAN output
CODE_DIR = os.path.join(os.path.dirname(os.getcwd()), '..', 'code')
sys.path.insert(0, CODE_DIR)
import swan_runner

# Paths to SWAN output files
MODEL_DIR = os.path.join(os.path.dirname(os.getcwd()), '..', 'california_bay.F')
TABLE_FILE = os.path.join(MODEL_DIR, 'site_output.tab')
BLOCK_FILE = os.path.join(MODEL_DIR, 'ga_block.mat')
SITES_FILE = os.path.join(MODEL_DIR, 'candidate_sites.dat')

for label, fp in [('TABLE', TABLE_FILE), ('BLOCK', BLOCK_FILE), ('SITES', SITES_FILE)]:
    print(f"{label} file: {'OK' if os.path.exists(fp) else 'MISSING  <-- ' + fp}")

## 1. Temporal Visualization: Time Series at Candidate Sites

The TABLE output from the AHP-TOPSIS site-selection run contains hourly values of $H_s$, $T_p$, wave direction, and depth at 391 candidate locations across the domain for the full Jan–Feb 2024 period (1440 hours).

We select a few representative ocean sites (filtering out land points marked with $H_s = -9$) and plot their time series to show:
- **Storm events** as peaks in $H_s$
- **Spatial variability** — how different sites experience the same storm differently
- The relationship between **wave height** and **peak period**

### How SWAN TABLE output is structured

SWAN writes all sites sequentially for each timestep:
```
t0-site0, t0-site1, ..., t0-siteN, t1-site0, t1-site1, ...
```
So for 391 sites and 1440 timesteps, there are $391 \times 1440 = 563{,}040$ rows. We reshape this into a 2-D array `(n_times, n_sites)` for easy indexing.

In [ ]:
# --- Load site locations and read TABLE output ---
site_locs = np.loadtxt(SITES_FILE)
N_SITES = site_locs.shape[0]

table_data = swan_runner.read_table(TABLE_FILE)
print(f"TABLE variables: {list(table_data.keys())}")
print(f"Total rows: {len(table_data['Hsig'])}")

# Reshape into (n_times, n_sites)
n_total = len(table_data['Hsig'])
N_TIMES = n_total // N_SITES
print(f"Sites: {N_SITES}, Timesteps: {N_TIMES}")

Hs_all = table_data['Hsig'].reshape(N_TIMES, N_SITES)
Tp_all = table_data['RTpeak'].reshape(N_TIMES, N_SITES)
Dir_all = table_data['Dir'].reshape(N_TIMES, N_SITES)
Dep_all = table_data['Depth'].reshape(N_TIMES, N_SITES)

# Create time axis (hourly, Jan 1 - Feb 29 2024)
time_start = datetime.datetime(2024, 1, 1, 0, 0, 0)
times = [time_start + datetime.timedelta(hours=i) for i in range(N_TIMES)]

# Find valid ocean sites (mean Hs > 0, not land)
# SWAN uses -9 as sentinel value for land/dry cells
Hs_valid = np.where(Hs_all > 0, Hs_all, np.nan)
mean_hs_per_site = np.nanmean(Hs_valid, axis=0)
ocean_mask = ~np.isnan(mean_hs_per_site)
ocean_indices = np.where(ocean_mask)[0]
print(f"Valid ocean sites: {len(ocean_indices)} out of {N_SITES}")

# Pick 4 representative sites spread across the range of mean Hs values
ocean_mean_hs = mean_hs_per_site[ocean_mask]
sorted_ocean = ocean_indices[np.argsort(ocean_mean_hs)]
pick_indices = [
    sorted_ocean[0],                          # lowest Hs (sheltered)
    sorted_ocean[len(sorted_ocean) // 3],     # low-mid
    sorted_ocean[2 * len(sorted_ocean) // 3], # high-mid
    sorted_ocean[-1],                         # highest Hs (exposed)
]
print(f"Selected site indices: {pick_indices}")
for idx in pick_indices:
    lon, lat = site_locs[idx]
    print(f"  Site {idx}: mean Hs={mean_hs_per_site[idx]:.2f} m, "
          f"({lon:.3f}\u00b0E, {lat:.3f}\u00b0N)")

### Time Series: $H_s$ and $T_p$ at Selected Sites

The top panel shows significant wave height $H_s$ and the bottom panel shows peak wave period $T_p$. Each colored line represents a different site within the domain — from a sheltered nearshore location to a fully exposed offshore location.

Storm events appear as synchronized peaks across all sites, but with different amplitudes reflecting the degree of wave transformation (refraction, shoaling, breaking).

In [ ]:
labels = ['Sheltered', 'Low-mid', 'High-mid', 'Exposed']
colors = ['#2166ac', '#67a9cf', '#ef8a62', '#b2182b']

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 9), sharex=True)
fig.suptitle('SWAN Output \u2014 Time Series at Selected Sites (California Bay, Jan\u2013Feb 2024)',
             fontsize=14, fontweight='bold')

for k, (idx, label, color) in enumerate(zip(pick_indices, labels, colors)):
    hs = Hs_all[:, idx].copy()
    tp = Tp_all[:, idx].copy()
    # Replace SWAN no-data (-9) with NaN for clean plots
    hs[hs < 0] = np.nan
    tp[tp < 0] = np.nan

    lon, lat = site_locs[idx]
    ax1.plot(times, hs, color=color, lw=0.8, alpha=0.85,
             label=f'{label} ({lon:.2f}\u00b0, {lat:.2f}\u00b0)')
    ax2.plot(times, tp, color=color, lw=0.8, alpha=0.85, label=f'{label}')

ax1.set_ylabel('Significant Wave Height $H_s$ (m)', fontsize=11)
ax1.legend(loc='upper right', fontsize=9, framealpha=0.9)
ax1.grid(True, alpha=0.2, ls='--')
ax1.set_ylim(bottom=0)

ax2.set_ylabel('Peak Wave Period $T_p$ (s)', fontsize=11)
ax2.set_xlabel('Date', fontsize=11)
ax2.legend(loc='upper right', fontsize=9, framealpha=0.9)
ax2.grid(True, alpha=0.2, ls='--')
ax2.set_ylim(bottom=0)

ax2.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=mdates.MO))
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
ax2.xaxis.set_minor_locator(mdates.DayLocator())
fig.autofmt_xdate(rotation=30)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

## 2. Spatial Visualization: 2-D Wave Height Map with Direction Quivers

The BLOCK output from the GA optimization run provides the full 2-D wave field on the SWAN computational grid (217 × 121 cells) at 6-hourly intervals. We compute the **time-averaged** significant wave height $\overline{H_s}$ and overlay **mean wave direction quivers**.

This map reveals:
- **Wave refraction** patterns caused by bathymetric features
- **Wave shadow** zones behind headlands and islands
- **Energy hotspots** where waves focus due to bottom topography
- **Spatial gradients** in wave height across the domain

### SWAN Sentinel Values

SWAN uses special sentinel values for land/dry cells:
- $H_s$, $T_p$: `-9`
- Direction: `-999`
- Depth: `-99`

These must be converted to `NaN` before any statistical computation.

In [ ]:
# --- Read BLOCK output ---
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")  # suppress duplicate variable warnings
    mat = swan_runner.read_mat_block(BLOCK_FILE)

Xp = mat['Xp']   # (ny, nx) longitude grid
Yp = mat['Yp']   # (ny, nx) latitude grid
Hs_block = swan_runner.extract_block_variable(mat, 'Hsig')    # (nt, ny, nx)
Tp_block = swan_runner.extract_block_variable(mat, 'RTpeak')  # (nt, ny, nx)
Dir_block = swan_runner.extract_block_variable(mat, 'Dir')    # (nt, ny, nx)
Dep_block = swan_runner.extract_block_variable(mat, 'Depth')  # (nt, ny, nx)

print(f"BLOCK grid: {Xp.shape[1]} x {Xp.shape[0]} (nx x ny)")
print(f"Timesteps: {Hs_block.shape[0]}")
print(f"Lon range: {Xp.min():.3f} to {Xp.max():.3f}")
print(f"Lat range: {Yp.min():.3f} to {Yp.max():.3f}")

# Replace SWAN sentinel values with NaN
Hs_block = np.where(Hs_block < 0, np.nan, Hs_block)
Tp_block = np.where(Tp_block < 0, np.nan, Tp_block)
Dir_block = np.where(Dir_block < -900, np.nan, Dir_block)
Dep_block = np.where(Dep_block < -90, np.nan, Dep_block)

# Time-averaged fields
Hs_mean = np.nanmean(Hs_block, axis=0)
Tp_mean = np.nanmean(Tp_block, axis=0)

# Circular mean direction: theta_mean = atan2(mean(sin(theta)), mean(cos(theta)))
Dir_rad = np.deg2rad(Dir_block)
mean_dir = np.rad2deg(np.arctan2(
    np.nanmean(np.sin(Dir_rad), axis=0),
    np.nanmean(np.cos(Dir_rad), axis=0),
)) % 360

# Land mask (where all Hs values are NaN)
land_mask = np.isnan(Hs_mean)

print(f"\nMean Hs range (ocean): {np.nanmin(Hs_mean):.3f} \u2013 {np.nanmax(Hs_mean):.3f} m")
print(f"Ocean cells: {(~land_mask).sum()}, Land cells: {land_mask.sum()}")

### Spatial Map: Mean $H_s$ with Direction Quivers

The filled contour shows time-averaged $H_s$ (warmer colors = higher waves). Arrow quivers show the **mean wave propagation direction**. Land cells are shaded in tan. The quiver subsampling factor (`skip=8`) prevents visual clutter while preserving the spatial pattern.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))

# Mask land for clean plotting
Hs_plot = np.ma.masked_where(land_mask, Hs_mean)

levels = np.linspace(np.nanmin(Hs_mean[~land_mask]),
                     np.nanmax(Hs_mean[~land_mask]), 25)
cf = ax.contourf(Xp, Yp, Hs_plot, levels=levels, cmap='YlOrRd', extend='both')
cbar = plt.colorbar(cf, ax=ax, shrink=0.8, pad=0.02)
cbar.set_label('Mean $H_s$ (m)', fontsize=11)

# Land fill
ax.contourf(Xp, Yp, np.where(land_mask, 1, np.nan),
            levels=[0.5, 1.5], colors=['#c8b882'], zorder=2)
ax.contour(Xp, Yp, np.where(land_mask, 1, 0),
           levels=[0.5], colors='k', linewidths=0.8, zorder=3)

# Direction quivers (subsample for clarity)
angle = np.deg2rad(270 - mean_dir)
U = np.cos(angle)
V = np.sin(angle)
skip = 8
sl = (slice(None, None, skip), slice(None, None, skip))
ax.quiver(Xp[sl], Yp[sl], U[sl], V[sl],
          color='navy', scale=35, headwidth=3, headlength=4,
          alpha=0.8, zorder=4)

ax.set_title('Mean Significant Wave Height & Direction \u2014 California Bay (SWAN output)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Longitude (\u00b0E)', fontsize=11)
ax.set_ylabel('Latitude (\u00b0N)', fontsize=11)
ax.set_aspect('equal')
ax.grid(True, alpha=0.15, ls='--')
plt.tight_layout()
plt.show()

## 3. Wave Power Density Map

The wave power density $P$ (kW/m) is the key metric for wave energy resource assessment. It is computed from the time-averaged fields using the deep-water approximation:

$$\bar{P} = \frac{\rho g^2}{64\pi} \overline{H_s}^2 \cdot \overline{T_p} \quad \text{(kW/m)}$$

where $\rho = 1025$ kg/m$^3$ is seawater density and $g = 9.81$ m/s$^2$.

This map directly shows where in the domain the wave energy resource is richest — the primary input for WEC site selection.

In [ ]:
# --- Wave power density map ---
rho_g2 = 1025 * 9.81**2 / (64 * np.pi)
P_mean = rho_g2 * Hs_mean**2 * Tp_mean / 1000  # kW/m

fig, ax = plt.subplots(figsize=(12, 10))

P_plot = np.ma.masked_where(land_mask, P_mean)
vmax = np.nanpercentile(P_mean[~land_mask], 99)
levels = np.linspace(0, vmax, 25)
cf = ax.contourf(Xp, Yp, P_plot, levels=levels, cmap='inferno', extend='max')
cbar = plt.colorbar(cf, ax=ax, shrink=0.8, pad=0.02)
cbar.set_label('Mean Wave Power Density (kW/m)', fontsize=11)

# Land fill
ax.contourf(Xp, Yp, np.where(land_mask, 1, np.nan),
            levels=[0.5, 1.5], colors=['#c8b882'], zorder=2)
ax.contour(Xp, Yp, np.where(land_mask, 1, 0),
           levels=[0.5], colors='k', linewidths=0.8, zorder=3)

ax.set_title('Mean Wave Power Density \u2014 California Bay (SWAN output)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Longitude (\u00b0E)', fontsize=11)
ax.set_ylabel('Latitude (\u00b0N)', fontsize=11)
ax.set_aspect('equal')
ax.grid(True, alpha=0.15, ls='--')
plt.tight_layout()
plt.show()

print(f"Max mean wave power: {np.nanmax(P_mean):.1f} kW/m")
print(f"Domain-average wave power (ocean only): {np.nanmean(P_mean[~land_mask]):.1f} kW/m")

## 4. Instantaneous Snapshot: Storm Peak vs Calm

Time-averaged maps smooth out storm variability. An **instantaneous snapshot** at a single timestep reveals the wave field during a specific event — useful for identifying refraction patterns, shadow zones, and the spatial structure of individual storm systems.

Below we select:
- The timestep with the **highest** domain-average $H_s$ (peak storm conditions)
- The timestep with the **lowest** domain-average $H_s$ (calm period)

and plot them side-by-side using the same color scale for direct comparison.

In [ ]:
# Domain-average Hs at each timestep (ignoring land NaNs)
domain_avg_hs = np.nanmean(Hs_block, axis=(1, 2))

# Identify storm peak and calm period
t_storm = np.argmax(domain_avg_hs)
t_calm = np.argmin(domain_avg_hs)

# Create time labels (6-hourly from Jan 1 2024)
block_times = [datetime.datetime(2024, 1, 1) + datetime.timedelta(hours=6*i)
               for i in range(Hs_block.shape[0])]

print(f"Storm peak:  timestep {t_storm} ({block_times[t_storm]:%Y-%m-%d %H:%M}), "
      f"domain-avg Hs = {domain_avg_hs[t_storm]:.2f} m")
print(f"Calm period: timestep {t_calm} ({block_times[t_calm]:%Y-%m-%d %H:%M}), "
      f"domain-avg Hs = {domain_avg_hs[t_calm]:.2f} m")

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

vmax = np.nanmax(Hs_block[t_storm])  # same scale for both panels
skip = 8
sl = (slice(None, None, skip), slice(None, None, skip))

for ax, t_idx, title_tag in zip(axes, [t_calm, t_storm], ['Calm', 'Storm Peak']):
    Hs_snap = Hs_block[t_idx]
    Hs_snap_plot = np.ma.masked_where(land_mask, Hs_snap)

    levels = np.linspace(0, vmax, 25)
    cf = ax.contourf(Xp, Yp, Hs_snap_plot, levels=levels,
                     cmap='YlOrRd', extend='max')

    # Land
    ax.contourf(Xp, Yp, np.where(land_mask, 1, np.nan),
                levels=[0.5, 1.5], colors=['#c8b882'], zorder=2)
    ax.contour(Xp, Yp, np.where(land_mask, 1, 0),
               levels=[0.5], colors='k', linewidths=0.8, zorder=3)

    # Direction quivers for this timestep
    Dir_snap = Dir_block[t_idx]
    Dir_snap = np.where(Dir_snap < -900, np.nan, Dir_snap)
    angle_snap = np.deg2rad(270 - Dir_snap)
    U_snap = np.cos(angle_snap)
    V_snap = np.sin(angle_snap)
    ax.quiver(Xp[sl], Yp[sl], U_snap[sl], V_snap[sl],
              color='navy', scale=35, headwidth=3, headlength=4,
              alpha=0.7, zorder=4)

    ax.set_title(f'{title_tag} \u2014 {block_times[t_idx]:%d %b %Y %H:%M UTC}\n'
                 f'Domain-avg $H_s$ = {domain_avg_hs[t_idx]:.2f} m',
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('Longitude (\u00b0E)', fontsize=10)
    ax.set_ylabel('Latitude (\u00b0N)', fontsize=10)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.15, ls='--')

cbar = fig.colorbar(cf, ax=axes, shrink=0.8, pad=0.03)
cbar.set_label('$H_s$ (m)', fontsize=11)

fig.suptitle('Instantaneous Wave Height Snapshots \u2014 California Bay (SWAN output)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 5. Reconstructed Directional Wave Spectrum

SWAN solves the spectral action balance equation internally, but TABLE output only provides **bulk parameters** ($H_s$, $T_p$, direction). We can reconstruct an approximate directional spectrum using the **JONSWAP** frequency shape and a **$\cos^{2s}$** directional spreading function.

### JONSWAP Frequency Spectrum

$$S(f) = \alpha g^2 (2\pi)^{-4} f^{-5} \exp\left[-\frac{5}{4}\left(\frac{f_p}{f}\right)^4\right] \gamma^{\exp\left[-\frac{(f-f_p)^2}{2\sigma^2 f_p^2}\right]}$$

where $f_p = 1/T_p$, $\gamma = 3.3$ (peak enhancement), and $\sigma = 0.07$ for $f \leq f_p$, $\sigma = 0.09$ for $f > f_p$.

### Directional Spreading

$$D(\theta) = \cos^{2s}\left(\frac{\theta - \theta_0}{2}\right)$$

where $\theta_0$ is the mean direction from SWAN and $s$ controls the spreading width ($s=15$ for narrow swell).

The full directional spectrum is then $E(f, \theta) = S(f) \cdot D(\theta)$.

In [ ]:
# Select the site with the highest mean Hs for the spectral plot
site_idx = pick_indices[-1]  # the "Exposed" site

hs_ts = Hs_all[:, site_idx].copy()
tp_ts = Tp_all[:, site_idx].copy()
dir_ts = Dir_all[:, site_idx].copy()

# Mask exception values
valid = (hs_ts > 0) & (tp_ts > 0) & (dir_ts > -900)
hs_mean = np.mean(hs_ts[valid])
tp_mean = np.mean(tp_ts[valid])

# Circular mean direction
dir_rad = np.deg2rad(dir_ts[valid])
dir_mean = np.rad2deg(np.arctan2(np.mean(np.sin(dir_rad)),
                                  np.mean(np.cos(dir_rad)))) % 360

fp = 1.0 / tp_mean  # peak frequency
lon, lat = site_locs[site_idx]
print(f"Spectral plot for Site {site_idx} ({lon:.2f}\u00b0E, {lat:.2f}\u00b0N)")
print(f"  Mean Hs = {hs_mean:.2f} m, Tp = {tp_mean:.1f} s, Dir = {dir_mean:.0f}\u00b0")

# --- Construct JONSWAP directional spectrum ---
frequencies = np.linspace(0.03, 0.5, 100)
directions = np.linspace(0, 2 * np.pi, 120)
F, D = np.meshgrid(frequencies, directions)

# JONSWAP frequency spectrum
gamma = 3.3
sigma = np.where(F <= fp, 0.07, 0.09)
S_f = (F ** -5) * np.exp(-1.25 * (fp / F) ** 4) * \
      gamma ** np.exp(-0.5 * ((F - fp) / (sigma * fp)) ** 2)

# Scale to match Hs: integral of S(f) ~ Hs^2 / 16
S_f *= (hs_mean ** 2 / 16.0) / np.trapezoid(S_f[0, :], frequencies)

# cos^2s directional spreading
theta_mean = np.deg2rad(dir_mean)
s_spread = 15  # narrow swell
D_theta = np.cos(0.5 * (D - theta_mean)) ** (2 * s_spread)
D_theta /= np.trapezoid(D_theta[:, 0], directions)  # normalize

E = S_f * D_theta

# --- Polar plot ---
fig, ax = plt.subplots(subplot_kw={'projection': 'polar'}, figsize=(9, 9))
levels = np.linspace(0, np.nanmax(E), 25)
cf = ax.contourf(D, F, E, levels=levels, cmap='magma')
fig.colorbar(cf, ax=ax, pad=0.1, shrink=0.85,
             label='Spectral Density $E(f, \\theta)$ (m\u00b2/Hz/rad)')

ax.set_theta_zero_location('N')
ax.set_theta_direction(-1)
ax.set_title(
    f'Directional Wave Spectrum \u2014 Site {site_idx}\n'
    f'$H_s$ = {hs_mean:.2f} m, $T_p$ = {tp_mean:.1f} s, '
    f'Dir = {dir_mean:.0f}\u00b0',
    fontsize=12, fontweight='bold', pad=20)
ax.set_ylabel('Frequency (Hz)', labelpad=30)

plt.tight_layout()
plt.show()

## Summary

This notebook demonstrated five standard post-processing visualizations using **real SWAN output data** from the California Bay simulation (Jan–Feb 2024):

| # | Visualization | Data Source | Key Insight |
|---|--------------|-------------|-------------|
| 1 | **Time series** ($H_s$, $T_p$) | TABLE output (`site_output.tab`) | Storm events, spatial variability between sheltered and exposed sites |
| 2 | **Spatial map** (mean $H_s$ + direction) | BLOCK output (`ga_block.mat`) | Refraction patterns, wave shadow zones, energy hotspots |
| 3 | **Wave power density** map | Derived from BLOCK $\overline{H_s}$ and $\overline{T_p}$ | Resource assessment — where wave energy is richest |
| 4 | **Instantaneous snapshots** (storm vs calm) | Single BLOCK timesteps | Spatial structure of individual events vs time-averaged fields |
| 5 | **Directional spectrum** (JONSWAP) | TABLE parameters at selected site | Frequency–direction energy distribution |

### Key Functions Used

- `swan_runner.read_table()` — parses SWAN TABLE text output into a dictionary of arrays
- `swan_runner.read_mat_block()` — reads SWAN BLOCK `.MAT` binary output via `scipy.io.loadmat`
- `swan_runner.extract_block_variable()` — stacks timestep matrices into `(time, ny, nx)` arrays

### Notes

- SWAN uses sentinel values ($-9$ for $H_s$/$T_p$, $-999$ for direction) to mark land/dry cells — these are masked to `NaN` before plotting
- Wave direction is averaged using circular statistics: $\bar{\theta} = \arctan2(\overline{\sin\theta}, \overline{\cos\theta})$
- Wave power density: $P = \frac{\rho g^2}{64\pi} H_s^2 T_p$ with $\rho = 1025$ kg/m³